# NeuroQWERTY - validation stricte imbriquée des candidats XGBoost/ondelettes

Ce notebook reprend les meilleures pistes du notebook 08 et les évalue avec une sélection imbriquée stricte. Le choix entre représentations et modèles se fait dans les folds internes, puis la performance est mesurée sur les folds externes groupés par sujet.

## 0. Configuration

In [1]:
from pathlib import Path
import re
import time
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pywt
from xgboost import XGBClassifier

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
OUTER_SPLITS = 5
INNER_SPLITS = 5
WINDOW = 300
STRIDE = 150
MIN_LEN = 300
WAVELET = "db4"
WAVELET_LEVEL = 4

ROOT = Path.cwd()
if not (ROOT / "data" / "neuroqwerty-mit-csxpd-dataset-1.0.0").exists() and ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_ROOT = ROOT / "data" / "neuroqwerty-mit-csxpd-dataset-1.0.0"
print("ROOT", ROOT)


SAVE_FINAL_MODEL = False
PRIMARY_SELECTION_METRIC = "f1_macro"  # Alternative possible: "roc_auc"
STRICT_EXPERIMENTS = [
    "agg_timing_xgb",
    "seq_wavelet_xgb",
    "combined_xgb",
    "agg_timing_histgb",
    "combined_histgb",
]
print("Strict nested evaluation for notebook 08 candidates")


ROOT /home/leonard/UQAC/8INF934 - Atelier Pratique IA I/Hackaton/parkinson-detection
Strict nested evaluation for notebook 08 candidates


## 1. Chargement et nettoyage

In [2]:
KEY_COLUMNS = ["key", "hold_time", "release_time", "press_time"]
MOUSE_RE = re.compile(r'"?mouse.+"?', re.I)
LONG_META_RE = re.compile(r'"?(Shift.+|Alt.+|Control.+)"?', re.I)
BACKSPACE_RE = re.compile(r'"?BackSpace"?', re.I)
PUNCT_OR_SPACE_RE = re.compile(r'"?(space|comma|period|semicolon|slash|minus|equal|apostrophe|Return)"?', re.I)
LEFT_KEYS = set("qwertasdfgzxcvb")
RIGHT_KEYS = set("yuiophjklnm")


def load_raw(path):
    df = pd.read_csv(path, header=None, names=KEY_COLUMNS)
    df["key"] = df["key"].astype(str).str.strip().str.replace('"', "", regex=False)
    for column in ["hold_time", "release_time", "press_time"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def clean(df):
    d = df.dropna(subset=["hold_time", "release_time", "press_time"]).copy()
    key = d["key"].astype(str)
    keep = ~key.str.match(MOUSE_RE) & ~key.str.match(LONG_META_RE) & ~key.str.match(BACKSPACE_RE)
    d = d.loc[keep]
    d = d[(d.press_time > 0) & (d.release_time > 0) & (d.hold_time.between(0, 5))]
    d = d.sort_values("press_time").reset_index(drop=True)
    d["flight_time"] = d.press_time.diff()
    d.loc[d.flight_time < 0, "flight_time"] = np.nan
    d["is_space_punct"] = d.key.str.match(PUNCT_OR_SPACE_RE).astype(int)
    d["hand_left"] = d.key.str.lower().str[:1].isin(LEFT_KEYS).astype(int)
    d["hand_right"] = d.key.str.lower().str[:1].isin(RIGHT_KEYS).astype(int)
    d["hand_switch"] = (d.hand_left.diff().abs().fillna(0) > 0).astype(int)
    return d


def load_sessions():
    rows = []
    raws = {}
    for dataset in ["MIT-CS1PD", "MIT-CS2PD"]:
        gt = pd.read_csv(DATA_ROOT / dataset / f"GT_DataPD_{dataset}.csv")
        raw_dir = DATA_ROOT / dataset / f"data_{dataset}"
        for _, subject in gt.iterrows():
            for file_col in [column for column in gt.columns if column.startswith("file_")]:
                filename = subject.get(file_col)
                if pd.isna(filename) or not str(filename).strip():
                    continue
                session_uid = f"{dataset}_{int(subject.pID)}_{file_col}"
                raw_path = raw_dir / str(filename)
                cleaned = clean(load_raw(raw_path))
                raws[session_uid] = cleaned
                rows.append({
                    "session_uid": session_uid,
                    "dataset": dataset,
                    "pID": int(subject.pID),
                    "session_file": str(filename),
                    "session_id": file_col,
                    "label": int(bool(subject["gt"])),
                    "n_keys": len(cleaned),
                })
    return pd.DataFrame(rows), raws


sessions, raw_sessions = load_sessions()
print("sessions", len(sessions), "subjects", sessions.pID.nunique())
display(sessions.groupby("label").agg(sessions=("label", "size"), subjects=("pID", "nunique")))


sessions 116 subjects 85


,sessions,subjects
label,,
0,56,43
1,60,42


## 2. Construction des features

In [3]:
def safe_div(a, b):
    return np.nan if b is None or pd.isna(b) or abs(b) < 1e-12 else a / b


def entropy_binary(p):
    if pd.isna(p) or p <= 0 or p >= 1:
        return 0.0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


def clean_signal(values, target_len=WINDOW):
    arr = pd.Series(values, dtype="float64").replace([np.inf, -np.inf], np.nan)
    median = arr.median()
    if pd.isna(median):
        median = 0.0
    arr = arr.fillna(median).to_numpy(dtype=float)
    if len(arr) < target_len:
        arr = np.pad(arr, (0, target_len - len(arr)), mode="edge")
    return arr[:target_len]


def stats_features(values, prefix):
    x = clean_signal(values)
    diff = np.diff(x)
    out = {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_median": float(np.median(x)),
        f"{prefix}_iqr": float(np.quantile(x, 0.75) - np.quantile(x, 0.25)),
        f"{prefix}_q10": float(np.quantile(x, 0.10)),
        f"{prefix}_q90": float(np.quantile(x, 0.90)),
        f"{prefix}_cv": safe_div(float(np.std(x)), float(np.mean(x))),
        f"{prefix}_diff_mean": float(np.mean(diff)),
        f"{prefix}_diff_std": float(np.std(diff)),
        f"{prefix}_diff_abs_mean": float(np.mean(np.abs(diff))),
    }
    for lag in [1, 2, 5, 10]:
        if len(x) > lag and np.std(x[:-lag]) > 0 and np.std(x[lag:]) > 0:
            out[f"{prefix}_autocorr_lag{lag}"] = float(np.corrcoef(x[:-lag], x[lag:])[0, 1])
        else:
            out[f"{prefix}_autocorr_lag{lag}"] = 0.0
    return out


def fft_features(values, prefix):
    x = clean_signal(values)
    x = x - np.mean(x)
    spectrum = np.abs(np.fft.rfft(x)) ** 2
    total = float(np.sum(spectrum)) + 1e-12
    thirds = np.array_split(spectrum, 3)
    return {
        f"{prefix}_fft_energy_total": total,
        f"{prefix}_fft_energy_low_ratio": float(np.sum(thirds[0]) / total),
        f"{prefix}_fft_energy_mid_ratio": float(np.sum(thirds[1]) / total),
        f"{prefix}_fft_energy_high_ratio": float(np.sum(thirds[2]) / total),
        f"{prefix}_fft_peak_bin": float(np.argmax(spectrum)),
    }


def wavelet_features(values, prefix):
    x = clean_signal(values)
    x = x - np.mean(x)
    coeffs = pywt.wavedec(x, WAVELET, level=WAVELET_LEVEL)
    out = {}
    for idx, coeff in enumerate(coeffs):
        name = "approx" if idx == 0 else f"detail_{idx}"
        abs_coeff = np.abs(coeff)
        energy = float(np.sum(coeff ** 2))
        prob = abs_coeff / (float(np.sum(abs_coeff)) + 1e-12)
        entropy = float(-np.sum(prob * np.log2(prob + 1e-12)))
        out.update({
            f"{prefix}_dwt_{name}_mean": float(np.mean(coeff)),
            f"{prefix}_dwt_{name}_std": float(np.std(coeff)),
            f"{prefix}_dwt_{name}_abs_mean": float(np.mean(abs_coeff)),
            f"{prefix}_dwt_{name}_energy": energy,
            f"{prefix}_dwt_{name}_entropy": entropy,
        })
    return out


def aggregate_features(segment):
    hold = segment.hold_time.dropna()
    flight = segment.flight_time.dropna()
    duration = segment.release_time.max() - segment.press_time.min() if len(segment) else np.nan
    out = {
        "n_keystrokes": len(segment),
        "duration_sec": duration,
        "keys_per_min": safe_div(len(segment) * 60, duration),
        "hold_to_flight": safe_div(hold.mean(), flight.mean()),
        "long_hold_rate": (hold > hold.quantile(0.9)).mean() if len(hold) > 5 else np.nan,
        "long_flight_rate": (flight > 1.0).mean() if len(flight) > 5 else np.nan,
        "space_punct_rate": segment.is_space_punct.mean(),
        "left_rate": segment.hand_left.mean(),
        "right_rate": segment.hand_right.mean(),
        "hand_switch_rate": segment.hand_switch.mean(),
    }
    out["hand_entropy"] = entropy_binary(out["left_rate"])
    out.update(stats_features(hold, "hold"))
    out.update(stats_features(flight, "flight"))
    return out


def sequence_features(segment):
    out = {}
    for column, prefix in [("hold_time", "hold"), ("flight_time", "flight")]:
        values = segment[column]
        out.update(fft_features(values, prefix))
        out.update(wavelet_features(values, prefix))
    return out


def build_segment_features():
    rows = []
    for _, session in sessions.iterrows():
        raw = raw_sessions[session.session_uid]
        starts = list(range(0, max(len(raw) - MIN_LEN + 1, 0), STRIDE))
        for segment_id, start in enumerate(starts):
            segment = raw.iloc[start:min(start + WINDOW, len(raw))]
            if len(segment) < MIN_LEN:
                continue
            rows.append({
                **session.to_dict(),
                "segment_id": segment_id,
                "segment_start": start,
                "segment_len": len(segment),
                **aggregate_features(segment),
                **sequence_features(segment),
            })
    return pd.DataFrame(rows)


segment_features = build_segment_features()
print(segment_features.shape, "sessions", segment_features.session_uid.nunique(), "subjects", segment_features.pID.nunique())
display(segment_features.groupby("label").agg(segments=("label", "size"), sessions=("session_uid", "nunique"), subjects=("pID", "nunique")))


(951, 109) sessions 115 subjects 84


,segments,sessions,subjects
label,,,
0,494,55,42
1,457,60,42


## 3. Candidats évalués

In [4]:
META_COLS = ["session_uid", "dataset", "pID", "session_file", "session_id", "label", "n_keys", "segment_id", "segment_start", "segment_len"]
LAYOUT_FEATURES = ["space_punct_rate", "left_rate", "right_rate", "hand_switch_rate", "hand_entropy"]
all_numeric = [column for column in segment_features.columns if column not in META_COLS and pd.api.types.is_numeric_dtype(segment_features[column])]
aggregate_timing_features = [f for f in all_numeric if not ("_fft_" in f or "_dwt_" in f) and f not in LAYOUT_FEATURES]
sequence_features_only = [f for f in all_numeric if "_fft_" in f or "_dwt_" in f]
combined_features = aggregate_timing_features + sequence_features_only

print("aggregate_timing_features", len(aggregate_timing_features))
print("sequence_features_only", len(sequence_features_only))
print("combined_features", len(combined_features))


def model_factory(name):
    if name == "histgb":
        return HistGradientBoostingClassifier(max_iter=250, learning_rate=0.035, l2_regularization=0.1, random_state=RANDOM_STATE)
    if name == "svc":
        return SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", probability=True, random_state=RANDOM_STATE)
    if name == "xgb":
        return XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.8,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    raise ValueError(name)


def make_pipeline(model_name):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model_factory(model_name)),
    ])

EXPERIMENTS = [
    {"name": "agg_timing_histgb", "features": aggregate_timing_features, "model": "histgb"},
    {"name": "agg_timing_xgb", "features": aggregate_timing_features, "model": "xgb"},
    {"name": "seq_wavelet_xgb", "features": sequence_features_only, "model": "xgb"},
    {"name": "combined_histgb", "features": combined_features, "model": "histgb"},
    {"name": "combined_xgb", "features": combined_features, "model": "xgb"},
    {"name": "combined_svc", "features": combined_features, "model": "svc"},
]
[(e["name"], e["model"], len(e["features"])) for e in EXPERIMENTS]


EXPERIMENT_MAP = {experiment["name"]: experiment for experiment in EXPERIMENTS}
EXPERIMENTS = [EXPERIMENT_MAP[name] for name in STRICT_EXPERIMENTS]
[(e["name"], e["model"], len(e["features"])) for e in EXPERIMENTS]


aggregate_timing_features 34
sequence_features_only 60
combined_features 94


[('agg_timing_xgb', 'xgb', 34),
 ('seq_wavelet_xgb', 'xgb', 60),
 ('combined_xgb', 'xgb', 94),
 ('agg_timing_histgb', 'histgb', 34),
 ('combined_histgb', 'histgb', 94)]

## 4. Utilitaires de validation

In [5]:
def aggregate_segment_predictions(table, valid_idx, probabilities, threshold):
    tmp = table.iloc[valid_idx][["session_uid", "pID", "label"]].copy()
    tmp["proba"] = probabilities
    agg = tmp.groupby("session_uid").agg(
        label=("label", "first"),
        pID=("pID", "first"),
        proba_mean=("proba", "mean"),
        n_segments=("proba", "size"),
    ).reset_index()
    agg["pred"] = (agg.proba_mean >= threshold).astype(int)
    return agg


def safe_auc(y_true, scores):
    return roc_auc_score(y_true, scores) if len(np.unique(y_true)) == 2 else np.nan


def safe_ap(y_true, scores):
    return average_precision_score(y_true, scores) if len(np.unique(y_true)) == 2 else np.nan


def metrics_from_agg(agg):
    return {
        "accuracy": accuracy_score(agg.label, agg.pred),
        "balanced_accuracy": balanced_accuracy_score(agg.label, agg.pred),
        "f1_macro": f1_score(agg.label, agg.pred, average="macro"),
        "f1_binary": f1_score(agg.label, agg.pred),
        "precision_binary": precision_score(agg.label, agg.pred, zero_division=0),
        "recall_binary": recall_score(agg.label, agg.pred, zero_division=0),
        "roc_auc": safe_auc(agg.label, agg.proba_mean),
        "pr_auc": safe_ap(agg.label, agg.proba_mean),
    }


def inner_oof(features, model_name, train_subjects):
    train_table = segment_features[segment_features.pID.isin(train_subjects)].copy()
    X = train_table[features]
    y = train_table.label.astype(int)
    groups = train_table.pID.astype(str)
    cv = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = []
    for train_idx, valid_idx in cv.split(X, y, groups):
        clf = make_pipeline(model_name)
        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        proba = clf.predict_proba(X.iloc[valid_idx])[:, 1]
        oof.append(aggregate_segment_predictions(train_table, valid_idx, proba, threshold=0.5))
    return pd.concat(oof, ignore_index=True)


def choose_threshold(oof, thresholds=np.linspace(0.2, 0.8, 61)):
    rows = []
    for threshold in thresholds:
        tmp = oof.copy()
        tmp["pred"] = (tmp.proba_mean >= threshold).astype(int)
        rows.append({"threshold": threshold, **metrics_from_agg(tmp)})
    threshold_df = pd.DataFrame(rows)
    best = threshold_df.sort_values(["f1_macro", "balanced_accuracy"], ascending=False).iloc[0]
    return float(best.threshold), threshold_df


def evaluate_experiment(experiment):
    y_subject = sessions.groupby("pID").label.first()
    subjects = y_subject.index.to_numpy()
    labels = y_subject.to_numpy()
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    oof_rows = []
    start = time.time()
    for fold, (train_subject_idx, test_subject_idx) in enumerate(outer_cv.split(subjects, labels, groups=subjects), 1):
        train_subjects = set(subjects[train_subject_idx])
        test_subjects = set(subjects[test_subject_idx])
        inner = inner_oof(experiment["features"], experiment["model"], train_subjects)
        threshold, _ = choose_threshold(inner)
        train_table = segment_features[segment_features.pID.isin(train_subjects)].copy()
        test_table = segment_features[segment_features.pID.isin(test_subjects)].copy()
        clf = make_pipeline(experiment["model"])
        clf.fit(train_table[experiment["features"]], train_table.label.astype(int))
        proba = clf.predict_proba(test_table[experiment["features"]])[:, 1]
        agg = aggregate_segment_predictions(test_table, np.arange(len(test_table)), proba, threshold=threshold)
        metrics = metrics_from_agg(agg)
        rows.append({
            "experiment": experiment["name"],
            "model": experiment["model"],
            "outer_fold": fold,
            "n_features": len(experiment["features"]),
            "threshold": threshold,
            **metrics,
        })
        oof_rows.append(agg.assign(experiment=experiment["name"], outer_fold=fold))
        print(
            f"{experiment['name']} fold {fold}/{OUTER_SPLITS}: "
            f"f1={metrics['f1_macro']:.3f} auc={metrics['roc_auc']:.3f} "
            f"threshold={threshold:.2f} elapsed={time.time() - start:.0f}s",
            flush=True,
        )
    return pd.DataFrame(rows), pd.concat(oof_rows, ignore_index=True)


all_results = []
all_oof = []
for experiment in EXPERIMENTS:
    results, oof = evaluate_experiment(experiment)
    all_results.append(results)
    all_oof.append(oof)

results_df = pd.concat(all_results, ignore_index=True)
oof_df = pd.concat(all_oof, ignore_index=True)
display(results_df)
summary = results_df.groupby("experiment")[["accuracy", "balanced_accuracy", "f1_macro", "f1_binary", "precision_binary", "recall_binary", "roc_auc", "pr_auc"]].agg(["mean", "std"]).round(3)
display(summary.sort_values(("roc_auc", "mean"), ascending=False))


agg_timing_xgb fold 1/5: f1=0.727 auc=0.843 threshold=0.51 elapsed=8s
agg_timing_xgb fold 2/5: f1=0.743 auc=0.818 threshold=0.78 elapsed=10s
agg_timing_xgb fold 3/5: f1=0.607 auc=0.786 threshold=0.59 elapsed=12s
agg_timing_xgb fold 4/5: f1=0.646 auc=0.892 threshold=0.53 elapsed=14s
agg_timing_xgb fold 5/5: f1=0.809 auc=0.833 threshold=0.51 elapsed=15s
seq_wavelet_xgb fold 1/5: f1=0.725 auc=0.818 threshold=0.56 elapsed=2s
seq_wavelet_xgb fold 2/5: f1=0.875 auc=0.881 threshold=0.71 elapsed=4s
seq_wavelet_xgb fold 3/5: f1=0.627 auc=0.720 threshold=0.58 elapsed=6s
seq_wavelet_xgb fold 4/5: f1=0.812 auc=0.883 threshold=0.56 elapsed=9s
seq_wavelet_xgb fold 5/5: f1=0.708 auc=0.741 threshold=0.66 elapsed=11s
combined_xgb fold 1/5: f1=0.817 auc=0.843 threshold=0.70 elapsed=3s
combined_xgb fold 2/5: f1=0.743 auc=0.860 threshold=0.58 elapsed=7s
combined_xgb fold 3/5: f1=0.575 auc=0.750 threshold=0.55 elapsed=10s
combined_xgb fold 4/5: f1=0.607 auc=0.800 threshold=0.67 elapsed=12s
combined_xgb fol

,experiment,model,outer_fold,n_features,threshold,accuracy,balanced_accuracy,f1_macro,f1_binary,precision_binary,recall_binary,roc_auc,pr_auc
0,agg_timing_xgb,xgb,1,34,0.51,0.727273,0.727273,0.727273,0.727273,0.727273,0.727273,0.842975,0.886544
1,agg_timing_xgb,xgb,2,34,0.78,0.750000,0.741259,0.742857,0.785714,0.733333,0.846154,0.818182,0.886787
2,agg_timing_xgb,xgb,3,34,0.59,0.653846,0.630952,0.606723,0.470588,0.800000,0.333333,0.785714,0.789248
3,agg_timing_xgb,xgb,4,34,0.53,0.681818,0.658333,0.645977,0.758621,0.647059,0.916667,0.891667,0.927778
4,agg_timing_xgb,xgb,5,34,0.51,0.809524,0.819444,0.809091,0.818182,0.900000,0.750000,0.833333,0.917989
5,seq_wavelet_xgb,xgb,1,60,0.56,0.727273,0.727273,0.725000,0.700000,0.777778,0.636364,0.818182,0.869687
6,seq_wavelet_xgb,xgb,2,60,0.71,0.875000,0.877622,0.874783,0.880000,0.916667,0.846154,0.881119,0.924112
7,seq_wavelet_xgb,xgb,3,60,0.58,0.653846,0.636905,0.626794,0.526316,0.714286,0.416667,0.720238,0.749076
8,seq_wavelet_xgb,xgb,4,60,0.56,0.818182,0.808333,0.811966,0.846154,0.785714,0.916667,0.883333,0.919983
9,seq_wavelet_xgb,xgb,5,60,0.66,0.714286,0.750000,0.708333,0.666667,1.000000,0.500000,0.740741,0.841906


accuracy        balanced_accuracy        f1_macro         \
                      mean    std              mean    std     mean    std   
experiment                                                                   
agg_timing_xgb       0.724  0.061             0.715  0.074    0.706  0.080   
agg_timing_histgb    0.691  0.084             0.681  0.098    0.668  0.107   
combined_histgb      0.708  0.090             0.697  0.101    0.688  0.106   
combined_xgb         0.735  0.107             0.729  0.122    0.720  0.125   
seq_wavelet_xgb      0.758  0.088             0.760  0.090    0.749  0.096   

                  f1_binary        precision_binary        recall_binary  \
                       mean    std             mean    std          mean   
experiment                                                                 
agg_timing_xgb        0.712  0.139            0.762  0.095         0.715   
agg_timing_histgb     0.689  0.131            0.735  0.123         0.698   
combined_histgb       0.703  0.144            0.742  0.116         0.715   
combined_xgb          0.720  0.162            0.783  0.158         0.698   
seq_wavelet_xgb       0.724  0.143            0.839  0.116         0.663   

                         roc_auc        pr_auc         
                     std    mean    std   mean    std  
experiment                                             
agg_timing_xgb     0.226   0.834  0.039  0.882  0.055  
agg_timing_histgb  0.210   0.826  0.024  0.873  0.039  
combined_histgb    0.219   0.825  0.033  0.872  0.066  
combined_xgb       0.210   0.814  0.043  0.869  0.070  
seq_wavelet_xgb    0.216   0.809  0.076  0.861  0.071

## 5. Sélection imbriquée stricte

Les folds internes choisissent la meilleure variante et le seuil. Le fold externe sert uniquement à mesurer la performance finale.

In [6]:
def summarize_inner_candidate(experiment, train_subjects):
    inner = inner_oof(experiment["features"], experiment["model"], train_subjects)
    threshold, threshold_df = choose_threshold(inner)
    tmp = inner.copy()
    tmp["pred"] = (tmp.proba_mean >= threshold).astype(int)
    metrics = metrics_from_agg(tmp)
    return threshold, metrics, threshold_df


def evaluate_outer_selected(experiment, train_subjects, test_subjects, threshold):
    train_table = segment_features[segment_features.pID.isin(train_subjects)].copy()
    test_table = segment_features[segment_features.pID.isin(test_subjects)].copy()
    clf = make_pipeline(experiment["model"])
    clf.fit(train_table[experiment["features"]], train_table.label.astype(int))
    probabilities = clf.predict_proba(test_table[experiment["features"]])[:, 1]
    agg = aggregate_segment_predictions(test_table, np.arange(len(test_table)), probabilities, threshold=threshold)
    return agg, metrics_from_agg(agg)


def run_nested_selection():
    y_subject = sessions.groupby("pID").label.first()
    subjects = y_subject.index.to_numpy()
    labels = y_subject.to_numpy()
    outer_cv = StratifiedGroupKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    selected_rows = []
    outer_rows = []
    oof_rows = []
    candidate_rows = []
    start = time.time()
    total_candidates = OUTER_SPLITS * len(EXPERIMENTS)
    done = 0
    print(
        f"Nested plan: outer_folds={OUTER_SPLITS}, candidates_per_fold={len(EXPERIMENTS)}, "
        f"total_candidate_evals={total_candidates}, selection_metric={PRIMARY_SELECTION_METRIC}",
        flush=True,
    )
    for outer_fold, (train_subject_idx, test_subject_idx) in enumerate(outer_cv.split(subjects, labels, groups=subjects), 1):
        fold_start = time.time()
        train_subjects = set(subjects[train_subject_idx])
        test_subjects = set(subjects[test_subject_idx])
        print("\n" + "=" * 96, flush=True)
        print(f"Outer fold {outer_fold}/{OUTER_SPLITS}: train_subjects={len(train_subjects)} test_subjects={len(test_subjects)}", flush=True)
        fold_candidates = []
        for experiment in EXPERIMENTS:
            candidate_start = time.time()
            done += 1
            print(f"  Candidate {done}/{total_candidates}: {experiment['name']} model={experiment['model']} n_features={len(experiment['features'])}", flush=True)
            threshold, inner_metrics, _ = summarize_inner_candidate(experiment, train_subjects)
            row = {
                "outer_fold": outer_fold,
                "experiment": experiment["name"],
                "model": experiment["model"],
                "n_features": len(experiment["features"]),
                "threshold": threshold,
                **{f"inner_{key}": value for key, value in inner_metrics.items()},
                "elapsed_sec": time.time() - candidate_start,
            }
            fold_candidates.append(row)
            candidate_rows.append(row)
            print(
                f"    inner_f1={inner_metrics['f1_macro']:.3f} inner_auc={inner_metrics['roc_auc']:.3f} "
                f"threshold={threshold:.2f} elapsed={row['elapsed_sec']:.0f}s",
                flush=True,
            )
        candidates_df = pd.DataFrame(fold_candidates)
        sort_cols = [f"inner_{PRIMARY_SELECTION_METRIC}", "inner_balanced_accuracy", "inner_roc_auc"]
        best_idx = candidates_df.sort_values(sort_cols, ascending=False).index[0]
        selected = candidates_df.loc[best_idx].to_dict()
        selected_experiment = next(e for e in EXPERIMENTS if e["name"] == selected["experiment"])
        agg, outer_metrics = evaluate_outer_selected(
            selected_experiment,
            train_subjects,
            test_subjects,
            threshold=float(selected["threshold"]),
        )
        selected_rows.append(selected)
        outer_rows.append({
            "outer_fold": outer_fold,
            "experiment": selected_experiment["name"],
            "model": selected_experiment["model"],
            "n_features": len(selected_experiment["features"]),
            "threshold": float(selected["threshold"]),
            **outer_metrics,
            "elapsed_sec": time.time() - fold_start,
        })
        oof_rows.append(agg.assign(outer_fold=outer_fold, experiment=selected_experiment["name"]))
        print(
            f"Selected fold {outer_fold}: {selected_experiment['name']} threshold={selected['threshold']:.2f} "
            f"outer_f1={outer_metrics['f1_macro']:.3f} outer_auc={outer_metrics['roc_auc']:.3f} "
            f"fold_elapsed={time.time() - fold_start:.0f}s",
            flush=True,
        )
    print(f"Total elapsed: {time.time() - start:.0f}s", flush=True)
    return (
        pd.DataFrame(outer_rows),
        pd.DataFrame(selected_rows),
        pd.DataFrame(candidate_rows),
        pd.concat(oof_rows, ignore_index=True),
    )


outer_results, selected_configs, candidate_results, outer_oof = run_nested_selection()
display(outer_results)
display(outer_results[["accuracy", "balanced_accuracy", "f1_macro", "f1_binary", "precision_binary", "recall_binary", "roc_auc", "pr_auc"]].agg(["mean", "std"]).round(3))
display(selected_configs[["outer_fold", "experiment", "model", "n_features", "threshold", "inner_f1_macro", "inner_roc_auc", "inner_pr_auc"]])


Nested plan: outer_folds=5, candidates_per_fold=5, total_candidate_evals=25, selection_metric=f1_macro

Outer fold 1/5: train_subjects=68 test_subjects=17
  Candidate 1/25: agg_timing_xgb model=xgb n_features=34
    inner_f1=0.753 inner_auc=0.793 threshold=0.51 elapsed=1s
  Candidate 2/25: seq_wavelet_xgb model=xgb n_features=60
    inner_f1=0.742 inner_auc=0.788 threshold=0.56 elapsed=2s
  Candidate 3/25: combined_xgb model=xgb n_features=94
    inner_f1=0.737 inner_auc=0.793 threshold=0.70 elapsed=3s
  Candidate 4/25: agg_timing_histgb model=histgb n_features=34
    inner_f1=0.731 inner_auc=0.795 threshold=0.57 elapsed=1s
  Candidate 5/25: combined_histgb model=histgb n_features=94
    inner_f1=0.731 inner_auc=0.784 threshold=0.60 elapsed=1s
Selected fold 1: agg_timing_xgb threshold=0.51 outer_f1=0.727 outer_auc=0.843 fold_elapsed=8s

Outer fold 2/5: train_subjects=68 test_subjects=17
  Candidate 6/25: agg_timing_xgb model=xgb n_features=34
    inner_f1=0.743 inner_auc=0.812 threshol

,outer_fold,experiment,model,n_features,threshold,accuracy,balanced_accuracy,f1_macro,f1_binary,precision_binary,recall_binary,roc_auc,pr_auc,elapsed_sec
0,1,agg_timing_xgb,xgb,34,0.51,0.727273,0.727273,0.727273,0.727273,0.727273,0.727273,0.842975,0.886544,8.365912
1,2,seq_wavelet_xgb,xgb,60,0.71,0.875000,0.877622,0.874783,0.880000,0.916667,0.846154,0.881119,0.924112,15.773006
2,3,combined_histgb,histgb,94,0.67,0.653846,0.630952,0.606723,0.470588,0.800000,0.333333,0.773810,0.762126,7.232999
3,4,agg_timing_xgb,xgb,34,0.53,0.681818,0.658333,0.645977,0.758621,0.647059,0.916667,0.891667,0.927778,8.624379
4,5,seq_wavelet_xgb,xgb,60,0.66,0.714286,0.750000,0.708333,0.666667,1.000000,0.500000,0.740741,0.841906,8.764076


,accuracy,balanced_accuracy,f1_macro,f1_binary,precision_binary,recall_binary,roc_auc,pr_auc
mean,0.730,0.729,0.713,0.701,0.818,0.665,0.826,0.868
std,0.086,0.096,0.103,0.150,0.142,0.244,0.066,0.069


,outer_fold,experiment,model,n_features,threshold,inner_f1_macro,inner_roc_auc,inner_pr_auc
0,1,agg_timing_xgb,xgb,34,0.51,0.752574,0.792672,0.854826
1,2,seq_wavelet_xgb,xgb,60,0.71,0.798077,0.806093,0.866305
2,3,combined_histgb,histgb,94,0.67,0.820202,0.832317,0.893259
3,4,agg_timing_xgb,xgb,34,0.53,0.774194,0.819444,0.865945
4,5,seq_wavelet_xgb,xgb,60,0.66,0.770758,0.798007,0.857429


## 6. Résultats finaux

In [7]:
print(classification_report(outer_oof.label, outer_oof.pred, target_names=["Contrôle", "Parkinson"]))
cm = confusion_matrix(outer_oof.label, outer_oof.pred)
display(pd.DataFrame(cm, index=["Réel contrôle", "Réel Parkinson"], columns=["Prédit contrôle", "Prédit Parkinson"]))

summary = outer_results[["accuracy", "balanced_accuracy", "f1_macro", "f1_binary", "precision_binary", "recall_binary", "roc_auc", "pr_auc"]].agg(["mean", "std"]).round(3)
display(summary)

display(selected_configs.groupby(["experiment", "model", "n_features"]).size().reset_index(name="count").sort_values("count", ascending=False))

top_candidates = candidate_results.groupby("experiment")[["inner_f1_macro", "inner_roc_auc", "inner_pr_auc"]].agg(["mean", "std"]).round(3)
display(top_candidates.sort_values(("inner_f1_macro", "mean"), ascending=False))


              precision    recall  f1-score   support

    Contrôle       0.69      0.80      0.74        55
   Parkinson       0.78      0.67      0.72        60

    accuracy                           0.73       115
   macro avg       0.74      0.73      0.73       115
weighted avg       0.74      0.73      0.73       115



,Prédit contrôle,Prédit Parkinson
Réel contrôle,44,11
Réel Parkinson,20,40


,accuracy,balanced_accuracy,f1_macro,f1_binary,precision_binary,recall_binary,roc_auc,pr_auc
mean,0.730,0.729,0.713,0.701,0.818,0.665,0.826,0.868
std,0.086,0.096,0.103,0.150,0.142,0.244,0.066,0.069


,experiment,model,n_features,count
0,agg_timing_xgb,xgb,34,2
2,seq_wavelet_xgb,xgb,60,2
1,combined_histgb,histgb,94,1


inner_f1_macro        inner_roc_auc        inner_pr_auc  \
                            mean    std          mean    std         mean   
experiment                                                                  
seq_wavelet_xgb            0.772  0.022         0.804  0.012        0.864   
combined_xgb               0.761  0.030         0.809  0.017        0.866   
agg_timing_xgb             0.760  0.019         0.803  0.015        0.858   
combined_histgb            0.758  0.040         0.801  0.031        0.854   
agg_timing_histgb          0.744  0.014         0.795  0.022        0.848   

                          
                     std  
experiment                
seq_wavelet_xgb    0.013  
combined_xgb       0.018  
agg_timing_xgb     0.017  
combined_histgb    0.027  
agg_timing_histgb  0.021

## 7. Export optionnel

In [8]:
if SAVE_FINAL_MODEL:
    config_counts = selected_configs.groupby(["experiment", "model", "n_features"]).size().reset_index(name="count").sort_values("count", ascending=False)
    best_name = config_counts.iloc[0].experiment
    best_experiment = next(experiment for experiment in EXPERIMENTS if experiment["name"] == best_name)
    all_subjects = set(sessions.pID.unique())
    inner = inner_oof(best_experiment["features"], best_experiment["model"], all_subjects)
    threshold, _ = choose_threshold(inner)
    final_model = make_pipeline(best_experiment["model"])
    final_model.fit(segment_features[best_experiment["features"]], segment_features.label.astype(int))
    artifact = {
        "pipeline": final_model,
        "features": best_experiment["features"],
        "model": best_experiment["model"],
        "experiment": best_experiment["name"],
        "level": "segment_mean_agg",
        "threshold": float(threshold),
        "min_segment_len": MIN_LEN,
        "window": WINDOW,
        "stride": STRIDE,
        "note": "Strict nested selection among notebook 08 XGBoost/wavelet candidates.",
    }
    path = ROOT / "models" / "keyboard_dynamics_neuroqwerty_auc_wavelet_xgboost_selected.joblib"
    import joblib
    joblib.dump(artifact, path)
    print(path)
else:
    print("No model exported. Set SAVE_FINAL_MODEL=True after reviewing the nested strict results.")


No model exported. Set SAVE_FINAL_MODEL=True after reviewing the nested strict results.


## 8. Analyse des résultats

La sélection imbriquée stricte donne une performance moyenne externe de `0.713 ± 0.103` en F1 macro, `0.729 ± 0.096` en balanced accuracy, `0.826 ± 0.066` en ROC-AUC et `0.868 ± 0.069` en PR-AUC.

Les configurations sélectionnées par fold sont :

| Fold | Configuration sélectionnée | F1 macro externe | ROC-AUC externe |
|---:|---|---:|---:|
| 1 | `agg_timing_xgb` | `0.727` | `0.843` |
| 2 | `seq_wavelet_xgb` | `0.875` | `0.881` |
| 3 | `combined_histgb` | `0.607` | `0.774` |
| 4 | `agg_timing_xgb` | `0.646` | `0.892` |
| 5 | `seq_wavelet_xgb` | `0.708` | `0.741` |

La sélection retient donc `agg_timing_xgb` dans 2 folds, `seq_wavelet_xgb` dans 2 folds et `combined_histgb` dans 1 fold. Il n’y a pas encore de configuration unique clairement dominante.

Le résultat est légèrement inférieur au meilleur candidat fixe du notebook 08 (`seq_wavelet_xgb`, `0.749 ± 0.096` en F1 macro). C’est normal : dans le notebook 08, on regarde après coup quel candidat a le mieux marché sur les folds externes. Dans le notebook 09, le candidat est choisi uniquement avec les folds internes. Le fold 3 illustre bien le problème : l’interne choisit `combined_histgb`, mais sa performance externe tombe à `0.607` en F1 macro.

Cela montre que les nouvelles représentations sont prometteuses, mais que le choix entre `agg_timing_xgb`, `seq_wavelet_xgb` et `combined_*` reste instable. Les scores internes des candidats sont très proches : `seq_wavelet_xgb` a le meilleur F1 interne moyen (`0.772`), mais `combined_xgb` et `agg_timing_xgb` suivent de très près (`0.761` et `0.760`). Avec seulement 84 sujets représentés après segmentation, de petites variations de folds peuvent changer le candidat retenu.

La matrice de confusion agrégée de la sélection stricte est :

| Classe réelle | Prédit contrôle | Prédit Parkinson |
|---|---:|---:|
| Contrôle | 44 | 11 |
| Parkinson | 20 | 40 |

La précision Parkinson est bonne (`0.78`), mais le rappel Parkinson est plus modéré (`0.67`). Le modèle sélectionné fait donc moins de faux positifs que certaines variantes précédentes, mais manque encore une partie des sujets Parkinson.

Conclusion : l’analyse stricte confirme une amélioration par rapport aux notebooks 06/07, mais elle réduit l’optimisme du notebook 08. La performance réaliste de cette famille de méthodes est plutôt autour de `0.71` en F1 macro et `0.83` en ROC-AUC. Pour l’application, il serait prématuré d’exporter automatiquement le modèle sélectionné. Une décision pragmatique serait de tester aussi deux stratégies fixes : `seq_wavelet_xgb` si l’objectif est le F1, et `agg_timing_xgb` si l’objectif est un score de risque continu pour la fusion.
